# XGBoost Models (All WorkCodes)
Compare with/without distance + block size analysis

In [1]:

import pandas as pd
import numpy as np
import time

from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from feature_engineer import get_engineered_df_allWC

In [2]:

DATA_PATH = "../data/processed/oe_detailed.parquet"
WAREHOUSE = "OE"
MAX_TIME = 300

df, features_all, cat_cols = get_engineered_df_allWC(
    file_path=DATA_PATH,
    warehouse=WAREHOUSE,
    max_time=MAX_TIME
)

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["date"] = df["Timestamp"].dt.date

print(df.shape)

(90714, 41)


In [3]:

distance_features = ["Travel_Distance", "log_travel_distance", "same_aisle", "same_location", "diff_level"]

features_with_dist = features_all
features_no_dist = [f for f in features_all if f not in distance_features]

cat_cols_no_dist = [c for c in cat_cols if c not in ["same_aisle", "same_location", "diff_level"]]

In [4]:

def split_chrono(df, ratio=0.15):
    days = sorted(df["date"].unique())
    n_test = max(1, int(len(days)*ratio))
    test_days = days[-n_test:]
    train = df[df["date"] < test_days[0]].copy()
    test = df[df["date"].isin(test_days)].copy()
    return train, test

train_df, test_df = split_chrono(df)

y_train = train_df["Time_Delta_sec"]
y_test = test_df["Time_Delta_sec"]

In [5]:

def make_X(train, test, features, cat_cols):
    X_train = pd.get_dummies(train[features], columns=cat_cols, drop_first=True)
    X_test = pd.get_dummies(test[features], columns=cat_cols, drop_first=True)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    return X_train, X_test

In [6]:

results = []

for label, feats, cats in [
    ("with_distance", features_with_dist, cat_cols),
    ("no_distance", features_no_dist, cat_cols_no_dist)
]:
    X_train, X_test = make_X(train_df, test_df, feats, cats)

    t0 = time.time()
    model = XGBRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    runtime = time.time() - t0

    results.append({
        "Model": label,
        "R2": r2_score(y_test, pred),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "Runtime": runtime
    })

pd.DataFrame(results)

,Model,R2,MAE,RMSE,Runtime
0,with_distance,0.520595,22.162223,35.763299,1.051709
1,no_distance,0.374978,26.070153,40.835126,1.106786


In [ ]:
def make_blocks(df, block_size):
    df = df.sort_values(["UserID", "Timestamp"])
    blocks = []
    for (uid, d), g in df.groupby(["UserID","date"]):
        for i in range(0, len(g), block_size):
            chunk = g.iloc[i:i+block_size]
            if len(chunk)==block_size:
                blocks.append(chunk)
    return blocks

In [ ]:
# ----------------------------
# Train model ONCE
# ----------------------------
X_train, X_test = make_X(train_df, test_df, features_no_dist, cat_cols_no_dist)

model = XGBRegressor(
    n_estimators=800,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# attach predictions safely
test_df = test_df.copy()
test_df["pred"] = model.predict(X_test)


# ----------------------------
# Block evaluation
# ----------------------------
block_sizes = [10, 20, 30, 40, 50]
block_results = []

for bs in block_sizes:
    blocks = make_blocks(test_df, bs)

    if len(blocks) == 0:
        print(f"No valid blocks for size {bs}")
        continue

    actual = []
    pred = []

    for b in blocks:
        actual.append(b["Time_Delta_sec"].sum())
        pred.append(b["pred"].sum()) 

    block_results.append({
        "BlockSize": bs,
        "NumBlocks": len(blocks),
        "MAE": mean_absolute_error(actual, pred),
        "MAE_per_task": mean_absolute_error(actual, pred) / bs
    })

In [18]:
# MAE and MAE_per_task to 3 digits
for r in block_results:
    r["MAE"] = round(r["MAE"], 3)

pd.DataFrame(block_results).sort_values("BlockSize")

,BlockSize,NumBlocks,MAE,MAE_per_task
0,10,1020,171.090,17.108979
1,20,502,298.519,14.925931
2,30,332,419.197,13.973237
3,40,246,519.623,12.990568
4,50,189,630.000,12.600000
